# 02 — Data Cleaning + SQL Database Design
## Mutual Fund Analytics | Day 2
---
**Objective:** Clean all 10 CSV datasets, design SQLite star schema, load data into database, write 10 analytical SQL queries, and create data dictionary.

## ⚙️ Setup

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import os
from pathlib import Path

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(f"📁 Working directory: {os.getcwd()}")

raw_path       = Path("data/raw/")
processed_path = Path("data/processed/")
processed_path.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete")


## 🧹 Step 1 — Clean `nav_history.csv`
Parse dates, sort, remove duplicates, forward-fill weekends/holidays, validate NAV > 0

In [ ]:
print("📄 Cleaning nav_history.csv ...")
nav = pd.read_csv(raw_path / "02_nav_history.csv")
nav["date"] = pd.to_datetime(nav["date"])
nav = nav.sort_values(["amfi_code", "date"]).reset_index(drop=True)

before = len(nav)
nav = nav.drop_duplicates(subset=["amfi_code", "date"])
print(f"   Duplicates removed : {before - len(nav)}")

invalid_nav = nav[nav["nav"] <= 0]
print(f"   Invalid NAV (<=0)  : {len(invalid_nav)}")
nav = nav[nav["nav"] > 0]

# Forward-fill missing NAV for holidays/weekends
all_filled = []
for code in nav["amfi_code"].unique():
    df_code = nav[nav["amfi_code"] == code].set_index("date")
    full_range = pd.date_range(df_code.index.min(), df_code.index.max(), freq="D")
    df_code = df_code.reindex(full_range)
    df_code["nav"] = df_code["nav"].ffill()
    df_code["amfi_code"] = code
    df_code = df_code.reset_index()
    df_code.columns = ["date", "amfi_code", "nav"]
    all_filled.append(df_code)

nav = pd.concat(all_filled, ignore_index=True)
nav = nav[["amfi_code", "date", "nav"]]
nav.to_csv(processed_path / "02_nav_history.csv", index=False)
print(f"   ✅ Saved — {len(nav):,} records (expanded with forward-fill)")


## 🧹 Step 2 — Clean `investor_transactions.csv`
Standardise transaction types, validate amounts, fix dates, check KYC status

In [ ]:
print("📄 Cleaning investor_transactions.csv ...")
txn = pd.read_csv(raw_path / "08_investor_transactions.csv")
txn["transaction_date"] = pd.to_datetime(txn["transaction_date"])
txn["transaction_type"] = txn["transaction_type"].str.strip().str.title()

valid_types = ["Sip", "Lumpsum", "Redemption"]
invalid_types = txn[~txn["transaction_type"].isin(valid_types)]["transaction_type"].unique()
print(f"   Invalid transaction types : {invalid_types if len(invalid_types) > 0 else 'None ✅'}")

before = len(txn)
txn = txn[txn["amount_inr"] > 0]
print(f"   Invalid amounts removed   : {before - len(txn)}")

valid_kyc = ["Verified", "Pending", "Rejected"]
invalid_kyc = txn[~txn["kyc_status"].isin(valid_kyc)]["kyc_status"].unique()
print(f"   Invalid KYC values        : {invalid_kyc if len(invalid_kyc) > 0 else 'None ✅'}")

txn = txn.drop_duplicates().reset_index(drop=True)
txn.to_csv(processed_path / "08_investor_transactions.csv", index=False)
print(f"   ✅ Saved — {len(txn):,} records")


## 🧹 Step 3 — Clean `scheme_performance.csv`
Validate return values, check expense ratio range (0.1%–2.5%), flag anomalies

In [ ]:
print("📄 Cleaning scheme_performance.csv ...")
perf = pd.read_csv(raw_path / "07_scheme_performance.csv")

return_cols = ["return_1yr_pct","return_3yr_pct","return_5yr_pct",
               "benchmark_3yr_pct","alpha","beta","sharpe_ratio",
               "sortino_ratio","std_dev_ann_pct","max_drawdown_pct"]
for col in return_cols:
    perf[col] = pd.to_numeric(perf[col], errors="coerce")

out_of_range = perf[(perf["expense_ratio_pct"] < 0.1) | (perf["expense_ratio_pct"] > 2.5)]
print(f"   Expense ratio out of range (0.1–2.5%) : {len(out_of_range)}")
anomalies = perf[(perf["return_1yr_pct"] > 100) | (perf["return_1yr_pct"] < -50)]
print(f"   Return anomalies flagged              : {len(anomalies)}")

perf = perf.drop_duplicates().reset_index(drop=True)
perf.to_csv(processed_path / "07_scheme_performance.csv", index=False)
print(f"   ✅ Saved — {len(perf):,} records")


## 🧹 Step 4 — Clean Remaining 7 Files

In [ ]:
other_files = {
    "01_fund_master.csv":          "launch_date",
    "03_aum_by_fund_house.csv":    "date",
    "04_monthly_sip_inflows.csv":  "month",
    "05_category_inflows.csv":     "month",
    "06_industry_folio_count.csv": "month",
    "09_portfolio_holdings.csv":   "portfolio_date",
    "10_benchmark_indices.csv":    "date",
}

for filename, date_col in other_files.items():
    df = pd.read_csv(raw_path / filename)
    df[date_col] = pd.to_datetime(df[date_col])
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    df.to_csv(processed_path / filename, index=False)
    print(f"   ✅ {filename:40s} → {len(df):,} rows | Duplicates removed: {before - len(df)}")

print("\n✅ All 10 files cleaned → data/processed/")


## 🗄️ Step 5 — Design & Load SQLite Star Schema

In [ ]:
DB_PATH = "data/db/bluestock_mf.db"
os.makedirs("data/db", exist_ok=True)
engine  = create_engine(f"sqlite:///{DB_PATH}")

# Read and execute schema.sql
with open("sql/schema.sql", "r") as f:
    schema_sql = f.read()

with engine.connect() as conn:
    for statement in schema_sql.split(";"):
        stmt = statement.strip()
        if stmt:
            conn.execute(text(stmt))
    conn.commit()
print("✅ Schema created from sql/schema.sql")

# Load all cleaned tables
tables = {
    "dim_fund":          "01_fund_master.csv",
    "fact_nav":          "02_nav_history.csv",
    "fact_aum":          "03_aum_by_fund_house.csv",
    "fact_sip":          "04_monthly_sip_inflows.csv",
    "fact_category":     "05_category_inflows.csv",
    "fact_folio":        "06_industry_folio_count.csv",
    "fact_performance":  "07_scheme_performance.csv",
    "fact_transactions": "08_investor_transactions.csv",
    "fact_portfolio":    "09_portfolio_holdings.csv",
    "fact_benchmark":    "10_benchmark_indices.csv",
}

print("\n📥 Loading tables into SQLite...")
for table, filename in tables.items():
    df = pd.read_csv(processed_path / filename)
    df.to_sql(table, engine, if_exists="replace", index=False)
    print(f"   ✅ {table:25s} → {len(df):6,} rows")


## ✅ Step 6 — Verify Row Counts Match Source CSVs

In [ ]:
print("\n🔍 Verifying row counts...")
with engine.connect() as conn:
    for table, filename in tables.items():
        source_count = len(pd.read_csv(processed_path / filename))
        db_count     = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        match        = "✅" if db_count == source_count else "❌ MISMATCH"
        print(f"   {table:25s}: source={source_count:6,}  db={db_count:6,}  {match}")

print(f"\n✅ Database ready → {DB_PATH}")


## 📝 Step 7 — Run 10 Analytical SQL Queries

In [ ]:
queries = {
    "Top 5 funds by AUM": """
        SELECT fund_house, SUM(aum_crore) AS total_aum_crore
        FROM fact_aum GROUP BY fund_house
        ORDER BY total_aum_crore DESC LIMIT 5""",

    "Average NAV per month": """
        SELECT SUBSTR(date,1,7) AS month, ROUND(AVG(nav),2) AS avg_nav
        FROM fact_nav GROUP BY SUBSTR(date,1,7)
        ORDER BY month LIMIT 10""",

    "SIP YoY growth": """
        SELECT month, sip_inflow_crore, yoy_growth_pct
        FROM fact_sip WHERE yoy_growth_pct IS NOT NULL
        ORDER BY month LIMIT 10""",

    "Transactions by state": """
        SELECT state, COUNT(*) AS total_transactions, SUM(amount_inr) AS total_amount
        FROM fact_transactions GROUP BY state
        ORDER BY total_transactions DESC LIMIT 10""",

    "Funds with expense ratio < 1%": """
        SELECT scheme_name, fund_house, expense_ratio_pct
        FROM dim_fund WHERE expense_ratio_pct < 1.0
        ORDER BY expense_ratio_pct ASC""",

    "Best 3yr return vs benchmark": """
        SELECT scheme_name, return_3yr_pct, benchmark_3yr_pct,
               ROUND(return_3yr_pct - benchmark_3yr_pct,2) AS alpha
        FROM fact_performance ORDER BY return_3yr_pct DESC LIMIT 5""",

    "Transaction type breakdown": """
        SELECT transaction_type, COUNT(*) AS count, SUM(amount_inr) AS total
        FROM fact_transactions GROUP BY transaction_type""",

    "Top sectors by portfolio weight": """
        SELECT sector, ROUND(SUM(weight_pct),2) AS total_weight
        FROM fact_portfolio GROUP BY sector
        ORDER BY total_weight DESC LIMIT 5""",

    "NAV growth of bluechip funds": """
        SELECT n.amfi_code, f.scheme_name,
               ROUND(((MAX(n.nav)-MIN(n.nav))/MIN(n.nav))*100,2) AS growth_pct
        FROM fact_nav n JOIN dim_fund f ON n.amfi_code=f.amfi_code
        WHERE n.amfi_code IN (119551,120503,118632,119092,120841)
        GROUP BY n.amfi_code ORDER BY growth_pct DESC""",

    "Monthly inflows by category": """
        SELECT month, category, net_inflow_crore
        FROM fact_category ORDER BY month DESC LIMIT 12""",
}

with engine.connect() as conn:
    for title, query in queries.items():
        print(f"\n{'='*55}")
        print(f"  Query: {title}")
        print(f"{'='*55}")
        result = pd.read_sql(text(query), conn)
        print(result.to_string(index=False))

print("\n✅ All 10 SQL queries executed successfully")


---
*Day 2: Data Cleaning + SQL Database Design complete | Bluestock MF Analytics Internship*